[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ielfeqi-rgb/omni_engine/blob/main/notebooks/Omni_Engine_ATLR_Benchmark_Playground.ipynb)

# ⚡ Omni Engine v2.2.2 Masterclass & Empirical ATLR Benchmark
### **Interactive Laboratory: Sovereign Autonomous Software Engineering with Qwen2.5-Coder-7B**
*100% Offline • Zero Cloud Subscriptions • Direct In-Process C-FFI • Deterministic Cleanroom Sandboxing*

---

### 🎯 Welcome to the Omni Engine Interactive Playground!
> **Hardware Setup:** In Google Colab, select **`Runtime -> Change runtime type -> T4 GPU`** before running this notebook.

This notebook serves as both a **comprehensive educational masterclass** and an **empirical replication laboratory** for the benchmark documented in the [Omni Engine Master Dossier](https://github.com/ielfeqi-rgb/omni_engine/blob/main/docs/OMNI_ENGINE_V2_2_1_SOVEREIGN_MASTER_DOSSIER.md).

Here, you will explore:
1. How **Omni Engine** eliminates cloud API latency, monthly subscriptions, and token fees by dynamically linking in-process C-FFI bindings to open-weights models (`.gguf`).
2. Why **Topological DAG Planning (The Maestro Protocol)** prevents the autoregressive token loops that trapped local wrapper agents (such as Aider Local for 5.5 hours).
3. How to reproduce the **Atomic Transaction Ledger in Rust (ATLR)** challenge: synthesizing a 50-thread deadlock-free financial ledger packaged cleanly as `.tar.gz`, extracting it, and running `cargo test --test atlr_stress` live inside Google Colab!


---
## 📦 Module 1: Hardware Check & Standalone CUDA v2.2.2 Provisioning

> 💡 **The Core Architecture Principle:**
> Unlike typical AI wrapper agents that rely on Python microservice middleware (Ollama, vLLM, or LangChain), Omni Engine is a **monolithic, memory-safe Rust binary**. It directly links against upstream `llama.cpp` and `ggml-cuda` shared libraries.

> 🛠️ **Clean Extraction Note:**
> The official release archive extracts under `omni_engine_v2.2.2_linux_x86_64_cuda/bin/`. Using `--strip-components=1` places `bin/` directly at the working root so `./bin/omni_engine` executes immediately with zero compilation!


In [ ]:
import os

# 0. Inspect allocated NVIDIA T4 GPU
!nvidia-smi

# 1. Clone the official Omni Engine repository (idempotent if re-run)
if not os.path.exists("omni_engine"):
    !git clone https://github.com/ielfeqi-rgb/omni_engine.git
%cd omni_engine

# 2. Install Bubblewrap cleanroom helper and curl/wget (optional in Colab container)
!apt-get update -qq && apt-get install -y -qq bubblewrap wget curl

# 3. Download official v2.2.2 CUDA GPU release tarball and strip top folder into ./bin/
!curl -fsSL https://github.com/ielfeqi-rgb/omni_engine/releases/download/v2.2.2/omni_engine_v2.2.2_linux_x86_64_cuda.tar.gz | tar -xz --strip-components=1
!chmod +x bin/*

# 4. Configure runtime library paths & verify sovereign binary
os.environ["OMNI_ALLOW_UNSANDBOXED"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"{os.getcwd()}/bin:/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
!./bin/omni_engine --version

print("✅ Module 1 Complete: Tesla T4 GPU & Omni Engine v2.2.2 CUDA binary ready!")


---
## 🧠 Module 2: In-Process C-FFI Model Provisioning

> 💡 **Why In-Process C-FFI Outperforms Microservices:**
> Standard agent frameworks make HTTP JSON-RPC calls over localhost to a background server (like Ollama). For an agent emitting hundreds of speculative tokens and tool calls, HTTP serialization overhead adds **40ms to 80ms of latency per exchange**. Omni Engine loads model weights directly into process RAM/VRAM via C-FFI pointers, executing token forward-passes with **zero serialization penalty**.

> 🛠️ **Model Selection Economics:**
> We use **`Qwen2.5-Coder-7B-Instruct-Q4_K_M`** (~4.68 GB). This model fits comfortably inside the 15 GB VRAM of a free Google Colab Tesla T4 GPU and downloads over Colab's high-speed network in ~25–40 seconds.


In [ ]:
# Download Qwen2.5-Coder-7B Instruct (Q4_K_M GGUF) directly from Hugging Face
!mkdir -p models
!wget -c "https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/main/qwen2.5-coder-7b-instruct-q4_k_m.gguf" -O models/qwen2.5-coder-7b.gguf

print("✅ Module 2 Complete: Qwen2.5-Coder-7B GGUF model downloaded and ready in models/")


---
## 📜 Module 3: The Maestro Protocol & Declarative DAG Planning

> 💡 **Why Monolithic Prompts Fail & Why DAGs Win:**
> Single-prompt coding agents (like vanilla ReAct) ask the model to generate the entire multi-file project in one giant markdown response. When a small-to-medium model hallucinates an import or syntax error midway, it attempts to rewrite the entire codebase, quickly spiraling into an **infinite autoregressive loop**.
> 
> Omni Engine solves this mathematically through the **Maestro Protocol**:
> 1. The problem is decomposed into a **Directed Acyclic Graph (DAG)** of discrete subgoals.
> 2. Each DAG node has explicit **Symbol Contracts**, **Invariants**, and **Acceptance Tests**.
> 3. Kahn's algorithm computes topological checkpoints, ensuring dependencies are synthesized and audited before downstream modules are attempted.

### 🏦 The Challenge: Atomic Transaction Ledger in Rust (ATLR)
Our challenge is to architect a high-concurrency financial ledger under uncompromising invariants:
- **Zero Dependencies:** Pure Rust standard library only (`std`). No external crates (`tokio`, `parking_lot`, etc.).
- **Zero Unsafe:** `#![forbid(unsafe_code)]` enforced at the crate root.
- **Deadlock Prevention (Canonical Lock Ordering):** Always acquire account locks ordered by ID: `min(from, to)` before `max(from, to)`.
- **Atomic 1-Unit Fee & Rollback:** Fixed 1-unit fee deducted first. If remaining balance < transfer amount, rollback the fee cleanly with zero leakage.
- **Liquidity Conservation:** Total system liquidity must remain strictly invariant (**30,000 units** across 3 accounts + treasury).
- **50-Thread Thundering Herd Stress Test:** A concurrent simulation testing all invariants under heavy thread contention.


In [ ]:
import json

# Define the canonical 4-file Maestro Plan specification for ATLR
plan_data = {
    "project": "AtomicTransactionLedgerRust (ATLR)",
    "language": "rust",
    "concurrency_limit": 1,
    "zero_dependencies": True,
    "no_unsafe": True,
    "nodes": [
        {
            "id": 1,
            "entity": "src/models.rs",
            "kind": "CREATE",
            "responsibility": "Define strict std-only domain models: Account struct, TransferError enum, and audit helpers without external crates or unsafe code.",
            "depends_on": [],
            "symbols": {
                "enums": [
                    {
                        "name": "TransferError",
                        "variants": [
                            "AccountNotFound(u64)",
                            "InsufficientFunds { available: i64, required: i64 }",
                            "SameAccountTransfer(u64)",
                            "InvalidAmount(i64)",
                            "LockPoisoned"
                        ]
                    }
                ],
                "structs": [
                    {
                        "name": "Account",
                        "fields": [
                            {"name": "id", "type": "u64"},
                            {"name": "balance", "type": "i64"}
                        ],
                        "methods": [
                            {"name": "new", "params": ["id: u64", "balance: i64"], "returns": "Self"},
                            {"name": "id", "params": ["&self"], "returns": "u64"},
                            {"name": "balance", "params": ["&self"], "returns": "i64"}
                        ]
                    }
                ]
            },
            "invariants": [
                "Strictly zero external dependencies (std only).",
                "Strictly zero unsafe blocks (#![forbid(unsafe_code)]).",
                "Balances must be i64.",
                "Include full Debug, Clone, and PartialEq derives where appropriate."
            ],
            "acceptance_tests": [
                "Syntax and symbol contracts pass with zero errors."
            ]
        },
        {
            "id": 2,
            "entity": "src/ledger.rs",
            "kind": "CREATE",
            "responsibility": "Implement the core thread-safe FinancialLedger with canonical lock ordering for deadlock prevention, atomic 1-unit fee deduction with rollback on shortfall, and fee collection to preserve total system liquidity.",
            "depends_on": ["src/models.rs"],
            "symbols": {
                "structs": [
                    {
                        "name": "FinancialLedger",
                        "fields": [
                            {"name": "accounts", "type": "std::sync::RwLock<std::collections::HashMap<u64, std::sync::Arc<std::sync::Mutex<crate::models::Account>>>>"},
                            {"name": "fee_account_id", "type": "u64"}
                        ],
                        "methods": [
                            {"name": "new", "params": ["fee_account_id: u64"], "returns": "Self"},
                            {"name": "register_account", "params": ["&self", "account: crate::models::Account"], "returns": "Result<(), crate::models::TransferError>"},
                            {"name": "get_balance", "params": ["&self", "id: u64"], "returns": "Result<i64, crate::models::TransferError>"},
                            {"name": "total_system_liquidity", "params": ["&self"], "returns": "Result<i64, crate::models::TransferError>"},
                            {"name": "transfer", "params": ["&self", "from_id: u64", "to_id: u64", "amount: i64"], "returns": "Result<(), crate::models::TransferError>"}
                        ]
                    }
                ]
            },
            "invariants": [
                "DEADLOCK PREVENTION: Acquire account locks strictly ordered by account ID: min(from_id, to_id) first, then max(from_id, to_id).",
                "REJECT REFLEXIVE: If from_id == to_id, return Err(TransferError::SameAccountTransfer(from_id)) immediately.",
                "VALIDATE POSITIVE AMOUNT: amount must be > 0. Otherwise return Err(TransferError::InvalidAmount(amount)).",
                "THE FEE RULE: Fixed fee of 1 unit. Deduct 1 unit from sender first.",
                "THE ROLLBACK RULE: If balance after fee deduction < amount, restore the 1 unit fee back to sender and return Err(TransferError::InsufficientFunds).",
                "FEE PRESERVATION: The 1 unit fee must be credited to fee_account_id so total system liquidity remains strictly constant.",
                "ATOMICITY: The entire fee deduction, rollback check, and recipient credit must occur under the held locks.",
                "NO UNSAFE: 100% safe Rust std::sync primitives only."
            ],
            "acceptance_tests": [
                "Syntax and symbol contracts pass with zero errors.",
                "Single-thread transfer deducts fee and credits recipient."
            ]
        },
        {
            "id": 3,
            "entity": "src/lib.rs",
            "kind": "CREATE",
            "responsibility": "Export models and ledger modules, declaring strict safety attributes #![forbid(unsafe_code)] and #![deny(unused_must_use)].",
            "depends_on": ["src/models.rs", "src/ledger.rs"],
            "invariants": [
                "Include #![forbid(unsafe_code)] at top.",
                "Include #![deny(unused_must_use)] at top.",
                "pub mod models; pub mod ledger;"
            ],
            "acceptance_tests": [
                "Syntax and symbol contracts pass with zero errors."
            ]
        },
        {
            "id": 4,
            "entity": "tests/atlr_stress.rs",
            "kind": "CREATE",
            "responsibility": "Implement the 50-Thread Thundering Herd Stress Simulator between 3 accounts and fee treasury, asserting total liquidity conservation, zero negative balances, and zero deadlocks.",
            "depends_on": ["src/lib.rs", "src/models.rs", "src/ledger.rs"],
            "invariants": [
                "Import from crate name 'atlr' (e.g., use atlr::models::Account; use atlr::ledger::{FinancialLedger};).",
                "Initialize 3 accounts: Account 1 (10,000), Account 2 (10,000), Account 3 (10,000) -> Initial Liquidity = 30,000.",
                "Set Account 0 as the Fee Collector Treasury (Initial Balance = 0). Total initial liquidity = 30,000.",
                "Spawn 50 concurrent threads executing reciprocal transfers simultaneously.",
                "Assert all threads join without deadlocks (timeout bounded).",
                "Assert final total_system_liquidity() == 30,000 exactly.",
                "Assert no account balance is negative.",
                "Assert rollback test: a transfer exceeding balance fails, but leaves balance untouched without fee leakage."
            ],
            "acceptance_tests": [
                "cargo test --test atlr_stress passes 100% with 0 failures."
            ]
        }
    ]
}

with open("atlr_colab_plan.json", "w") as f:
    json.dump(plan_data, f, indent=2)

print("✅ Module 3 Complete: atlr_colab_plan.json generated successfully!")


---
## 🚀 Module 4: Autonomous Swarm Execution (In-Memory VFS -> Hermetic `.tar.gz`)

> 💡 **The Self-Healing Loop & Hermetic Packaging in Action:**
> When you execute the command below, Omni Engine v2.2.2 orchestrates the 4 subgoals:
> 1. **RAM Virtual Filesystem (`MemoryVfs`):** Workers write files strictly in RAM — never polluting the host repository (`run` mode).
> 2. **Causal KV Slicing:** If an invariant issue occurs, the engine rolls back the KV cache to the last topological checkpoint in ~6 milliseconds.
> 3. **Hermetic `.tar.gz` Commit Gate:** Verified deliverables are packaged cleanly into `atlr_workspace/omni_deliverables_<TIMESTAMP>.tar.gz` (zero loose file spill!).

> 🛠️ **Why We Install Cargo in Module 5:**
> In ATLR, `src/ledger.rs` (Node 2) references `crate::models`, while `src/lib.rs` (Node 3) is written afterward to wire `pub mod models; pub mod ledger;`. Running generation cleanly via `atlr_colab_plan.json` first and then compiling the extracted crate with `cargo test` in Module 5 matches the live Colab Tesla T4 benchmark pipeline.


In [ ]:
import os

os.makedirs("atlr_workspace", exist_ok=True)
os.environ["OMNI_ALLOW_UNSANDBOXED"] = "1"
os.environ["LD_LIBRARY_PATH"] = f"{os.getcwd()}/bin:/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"

# Execute Omni Engine with the ATLR Maestro Plan into isolated ./atlr_workspace
!./bin/omni_engine run "Implement ATLR financial ledger with zero dependencies and deadlock prevention" \
    --plan-file atlr_colab_plan.json \
    --output-dir ./atlr_workspace \
    --model models/qwen2.5-coder-7b.gguf \
    --subgoals 4 \
    --steps 3 \
    -y

!ls -lh atlr_workspace/
print("✅ Module 4 Complete: Omni Engine packaged deliverables cleanly in atlr_workspace/!")


---
## 🏆 Module 5: Extract Hermetic `.tar.gz` & Execute 50-Thread Cargo Stress Test Live

> 💡 **The Moment of Truth:**
> 1. Install the stable Rust & Cargo toolchain in Colab (if not already installed).
> 2. Extract the hermetic deliverable tarball produced by v2.2.2 (`atlr_workspace/omni_deliverables_*.tar.gz`).
> 3. Create `atlr_workspace/Cargo.toml` (`name = "atlr"`, strictly zero external dependencies).
> 4. Run `cargo test --test atlr_stress -- --nocapture` live on Google Colab asserting:
>    - **Zero Deadlocks:** All 50 concurrent threads join cleanly.
>    - **Zero Money Lost / Created:** Total liquidity remains exactly `30,000` units!
>    - **Zero Negative Balances:** Safe atomic 1-unit fee rollbacks.


In [ ]:
import glob
import os
import shutil
import subprocess

# 1. Install stable Rust & Cargo toolchain if not already installed
cargo_bin = os.path.expanduser("~/.cargo/bin")
if cargo_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"

if shutil.which("cargo") is None:
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

!cargo --version && rustc --version

# 2. Extract v2.2.2 deliverable tarball into atlr_workspace/
archives = sorted(glob.glob("atlr_workspace/omni_deliverables_*.tar.gz")) or sorted(glob.glob("atlr_workspace/omni_rescue_material_*.tar.gz"))
assert archives, "No Omni Engine .tar.gz archive found in atlr_workspace/"
latest_archive = archives[-1]
print(f"📦 Extracting Sovereign Deliverable Archive: {latest_archive}")
subprocess.run(["tar", "-xzf", latest_archive, "-C", "atlr_workspace"], check=True)

# 3. Initialize strict std-only Cargo.toml
with open("atlr_workspace/Cargo.toml", "w") as f:
    f.write("""[package]
name = "atlr"
version = "0.1.0"
edition = "2021"

[dependencies]
# STRICTLY ZERO EXTERNAL DEPENDENCIES (std only)
""")

# 4. Execute the 50-Thread Thundering Herd Stress Test live with Cargo!
!cd atlr_workspace && cargo test --test atlr_stress -- --nocapture

print("🏆 SHOWDOWN COMPLETE: ATLR Benchmark 100% Verified Live on Google Colab!")


---
## 📊 Module 6: Empirical Results & Showdown Summary

```
+-------------------------------------------------------------------------------------------------------------------------------+
|                                                ATLR TRI-PARTY & SCALING BENCHMARK MATRIX                                      |
+-------------------+-----------------------------+------------------------+------------------------+---------------------------+
| Metric            | Omni Engine (Local 4B)      | Aider Local            | Aider Cloud            | Omni Engine (7B Tesla T4) |
+-------------------+-----------------------------+------------------------+------------------------+---------------------------+
| Executing Model   | Qwen3-4B-Q6_K (Local)       | Qwen3-4B (via Ollama)  | Gemini 2.5 Flash (API) | Qwen2.5-Coder-7B (Local)  |
| Compute Runtime   | Commodity CPU (8 cores)     | Commodity CPU (8 cores)| Google Cloud Servers   | NVIDIA Tesla T4 GPU       |
| Wall-Clock Time   | 1h 39m (99.5 mins)          | +5h 24m (Timed out)    | 58 seconds             | 37m 58s (37.97 mins)      |
| Speedup Factor    | 1.0x (Baseline)             | 0.0x (Catastrophic)    | Fast Cloud API         | 2.6x Speedup over 4B      |
| Process Exit Code | Exit Code: 0 (Clean)        | KILLED (`kill -9`)     | Exit Code: 0           | Exit Code: 0 (Pristine)   |
| Concurrency Logic | Correct (`min`/`max` locks) | N/A (Never reached)    | Correct (`sort_unstable`)| Correct (RwLock + Mutex) |
| Stress Outcome    | Staged safely in RAM VFS    | Complete Failure       | FAILED (Test Panicked) | 100% Passed Live in Colab |
| Cloud Dependency  | ZERO (100% Sovereign)       | Needs Ollama Daemon    | Proprietary Cloud API  | ZERO (100% Sovereign)     |
+-------------------+-----------------------------+------------------------+------------------------+---------------------------+
```

### 🌟 Key Takeaways
1. **Zero Infinite Loops:** Deterministic DAG decomposition prevents the autoregressive collapse that trapped Aider Local for 5.5 hours.
2. **Superior Concurrency Correctness:** Omni Engine avoided the logic fallacies that caused Gemini 2.5 Flash to panic under concurrent load.
3. **100% Sovereign & Offline:** Everything executed using open-weights `.gguf` models without sending a single byte to cloud APIs.

### 📚 Official Resources & Documentation
- **Official GitHub Repository:** [https://github.com/ielfeqi-rgb/omni_engine](https://github.com/ielfeqi-rgb/omni_engine)
- **Master Sovereign Dossier:** [`docs/OMNI_ENGINE_V2_2_1_SOVEREIGN_MASTER_DOSSIER.md`](https://github.com/ielfeqi-rgb/omni_engine/blob/main/docs/OMNI_ENGINE_V2_2_1_SOVEREIGN_MASTER_DOSSIER.md)
- **Technical Whitepaper:** [`docs/OMNI_ENGINE_TECHNICAL_WHITEPAPER.md`](https://github.com/ielfeqi-rgb/omni_engine/blob/main/docs/OMNI_ENGINE_TECHNICAL_WHITEPAPER.md)
- **Tri-Party Showdown Report:** [`docs/ATLR_TRI_PARTY_SHOWDOWN_REPORT.md`](https://github.com/ielfeqi-rgb/omni_engine/blob/main/docs/ATLR_TRI_PARTY_SHOWDOWN_REPORT.md)
- **ATLR Maestro Blueprint:** [`examples/atlr_maestro_plan.json`](https://github.com/ielfeqi-rgb/omni_engine/blob/main/examples/atlr_maestro_plan.json)
- **Dual-Licensing Inquiries:** [ielfeqi@gmail.com](mailto:ielfeqi@gmail.com)
